# Ejercicio 3 — Carga incremental que no se rompa con datos viejos
---

Necesitamos cargar inventory_silver de forma incremental usando updated_ts como watermark.  
Fíjate bien en la fila de P-100/HN en el batch — no es un error de dedo, es intencional.  
10. Define la configuración (merge_keys, columna de watermark) en un diccionario Python.  
11. Escribe incremental_load(spark, batch_df, target_table_path, conf): que revise si la tabla ya existe (si no, la crea), que respete el watermark para no pisar datos con algo más viejo, que inserte lo nuevo, y actualice solo cuando corresponde.  
12. Dinos cómo queda inventory_silver después de correr tu función contra estos datos — las 4 filas originales más la nueva.

### Imports


In [1]:
from src.common.spark_session import get_spark
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import shutil
import os

spark = get_spark("ejercicio3-incremental-load")
spark


:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5a4ec21c-b655-40af-a012-d37ce9fce362;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 166ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

---

## Paso 10: Configuración en diccionario Python

In [2]:
# Configuracion 
config = {
    "merge_keys":["producto_id", "pais_cd"],
    "watermark": "updated_ts"
}

## Paso 11: Función incremental_load()

In [3]:
# Función incremental_load
def incremental_load(spark, batch_df, target_table_path, conf):

    merge_keys = conf["merge_keys"]
    watermark = conf["watermark"]

    # Se deduplica el batch para obtener producto + pais mas reciente
    window = (
        Window.partitionBy(*merge_keys)
        .orderBy(F.col(watermark).desc())
    )

    batch_df = (
        batch_df
        .withColumn("rn", F.row_number().over(window))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    # Si la tabla delta no existe, se crea
    if not DeltaTable.isDeltaTable(spark, target_table_path):
        batch_df.write.format("delta").mode("overwrite").save(target_table_path)
        return batch_df

    # Si la tabla delta existe, se obtiene la tabla
    target = DeltaTable.forPath(spark, target_table_path)

    # Se realiza el merge de los datos
    (
        target.alias("t")
        .merge(
            batch_df.alias("s"),
            " AND ".join([f"t.{k} = s.{k}" for k in merge_keys])
        )
        .whenMatchedUpdateAll(condition=f"s.{watermark} > t.{watermark}")
        .whenNotMatchedInsertAll()
        .execute()
    )

    return target

## Paso 11: Verificación de la función incremental_load

In [4]:
# Ruta temporal de inventory_silver
target_table_path = "/tmp/inventory_silver_test"

In [5]:
# Tabla histórica inventory_silver actual
inventory_silver = [
    ("P-100", "GT", 45, "2024-05-01 08:00:00"),
    ("P-101", "GT", 12, "2024-05-01 08:00:00"),
    ("P-100", "HN", 20, "2024-05-01 08:00:00"),
    ("P-102", "AR", 8,  "2024-04-30 17:30:00")
]

historical_df = spark.createDataFrame(
    inventory_silver,
    ["producto_id", "pais_cd", "stock_int", "updated_ts"]
)

historical_df = historical_df.withColumn(
    "updated_ts",
    F.to_timestamp("updated_ts")
)

historical_df.show()

+-----------+-------+---------+-------------------+
|producto_id|pais_cd|stock_int|         updated_ts|
+-----------+-------+---------+-------------------+
|      P-100|     GT|       45|2024-05-01 08:00:00|
|      P-101|     GT|       12|2024-05-01 08:00:00|
|      P-100|     HN|       20|2024-05-01 08:00:00|
|      P-102|     AR|        8|2024-04-30 17:30:00|
+-----------+-------+---------+-------------------+



In [6]:
# Batch que llega hoy inventory_bronze_batch
inventory_bronze_batch = [
    ("P-100", "GT", 40, "2024-05-02 09:10:00"),
    ("P-103", "GT", 5,  "2024-05-02 09:12:00"),
    ("P-100", "HN", 20, "2024-04-29 10:00:00"),
    ("P-102", "AR", 3,  "2024-05-02 10:00:00")
]

batch_df = spark.createDataFrame(
    inventory_bronze_batch,
    ["producto_id", "pais_cd", "stock_int", "updated_ts"]
)

batch_df = batch_df.withColumn(
    "updated_ts",
    F.to_timestamp("updated_ts")
)

batch_df.show()

+-----------+-------+---------+-------------------+
|producto_id|pais_cd|stock_int|         updated_ts|
+-----------+-------+---------+-------------------+
|      P-100|     GT|       40|2024-05-02 09:10:00|
|      P-103|     GT|        5|2024-05-02 09:12:00|
|      P-100|     HN|       20|2024-04-29 10:00:00|
|      P-102|     AR|        3|2024-05-02 10:00:00|
+-----------+-------+---------+-------------------+



## Carga de inventory_silver con historial previo

In [7]:
# Verificacion si existe la ruta, si existe lo borra
if os.path.exists(target_table_path):
    shutil.rmtree(target_table_path)

In [8]:
# Guarda inventory_silver para tener historial
(
    historical_df.write
    .format("delta")
    .mode("overwrite")
    .save(target_table_path)
)

In [9]:
# Visualizacion inventory_silver desde tabla delta
if DeltaTable.isDeltaTable(spark, target_table_path):
    print("Inventory_silver delta existe. Leyendo...")
    
    spark.read.format("delta") \
        .load(target_table_path) \
        .show()

else:
    print("Inventory_silver delta no existe todavía.")

Inventory_silver delta existe. Leyendo...


26/08/17 18:15:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+-------+---------+-------------------+
|producto_id|pais_cd|stock_int|         updated_ts|
+-----------+-------+---------+-------------------+
|      P-100|     HN|       20|2024-05-01 08:00:00|
|      P-100|     GT|       45|2024-05-01 08:00:00|
|      P-101|     GT|       12|2024-05-01 08:00:00|
|      P-102|     AR|        8|2024-04-30 17:30:00|
+-----------+-------+---------+-------------------+



In [10]:
# Se aplica la funcion para incrementar inventory_silver
incremental_load(spark, batch_df, target_table_path, config)

In [11]:
# Visualización de inventory_silver ya incrementada
result_df = (
    spark.read
    .format("delta")
    .load(target_table_path)
)

print("Inventory_silver leido desde Delta")
result_df.orderBy("producto_id", "pais_cd").show(truncate=False)

Inventory_silver leido desde Delta
+-----------+-------+---------+-------------------+
|producto_id|pais_cd|stock_int|updated_ts         |
+-----------+-------+---------+-------------------+
|P-100      |GT     |40       |2024-05-02 09:10:00|
|P-100      |HN     |20       |2024-05-01 08:00:00|
|P-101      |GT     |12       |2024-05-01 08:00:00|
|P-102      |AR     |3        |2024-05-02 10:00:00|
|P-103      |GT     |5        |2024-05-02 09:12:00|
+-----------+-------+---------+-------------------+



## Carga de inventory_silver sin historial previo

In [12]:
# Verificacion si existe la ruta, si existe lo borra
if os.path.exists(target_table_path):
    shutil.rmtree(target_table_path)

In [13]:
# Visualizacion inventory_silver desde tabla delta
if DeltaTable.isDeltaTable(spark, target_table_path):
    print("Inventory_silver delta existe. Leyendo...")
    
    spark.read.format("delta") \
        .load(target_table_path) \
        .show()

else:
    print("Inventory_silver delta no existe todavía.")

Inventory_silver delta no existe todavía.


In [14]:
# Se aplica la funcion para incrementar inventory_silver
incremental_load(spark, batch_df, target_table_path, config)

DataFrame[producto_id: string, pais_cd: string, stock_int: bigint, updated_ts: timestamp]

In [15]:
# Visualización de inventory_silver ya incrementada
result_df = (
    spark.read
    .format("delta")
    .load(target_table_path)
)

print("Inventory_silver leido desde Delta")
result_df.orderBy("producto_id", "pais_cd").show(truncate=False)

Inventory_silver leido desde Delta
+-----------+-------+---------+-------------------+
|producto_id|pais_cd|stock_int|updated_ts         |
+-----------+-------+---------+-------------------+
|P-100      |GT     |40       |2024-05-02 09:10:00|
|P-100      |HN     |20       |2024-04-29 10:00:00|
|P-102      |AR     |3        |2024-05-02 10:00:00|
|P-103      |GT     |5        |2024-05-02 09:12:00|
+-----------+-------+---------+-------------------+

